In [1]:
import pandas as pd
import numpy as np
import pathlib
#import glob
#import os

import config

In [2]:
## Create FS Data per customer 
for customerName in config.customerList:
    for auditLog in config.auditList:
        DF = pd.read_parquet(f'{config.data_location}FS_base_data/{auditLog}/FS_base_data_{auditLog}_{customerName}.parquet', engine='pyarrow')
        DF.dropna(how='all',axis=1,inplace=True)
        dirPath = f'{config.data_location}FS_audit_type_table_name//{auditLog}//'
        pathlib.Path(dirPath).mkdir(parents=True, exist_ok=True)
        fileName = f'{dirPath}FS_audit_type_table_name_{auditLog}_{customerName}.parquet'
        DF.to_parquet(path=fileName)

        ## Create FS Data by audit type 
        FilterTypeDF = pd.read_parquet(f'{config.data_location}FS_base_data/{auditLog}/FS_base_data_{auditLog}_{customerName}.parquet', engine='pyarrow')
        FilterTypeList = FilterTypeDF['AUDIT_TYPE'].unique().tolist() 
        for FilterType in FilterTypeList:
            dirPath = f'{config.data_location}//FS_audit_type_table_name//{auditLog}//{FilterType}//'
            pathlib.Path(dirPath).mkdir(parents=True, exist_ok=True)       
            fileName = f'{dirPath}FS_audit_type_table_name_{auditLog}_{FilterType}_{customerName}.parquet' 
            try:
                AuditTypeDF = DF.query(f'AUDIT_TYPE=="{FilterType}"')

                AuditTypeDF.to_parquet(path=fileName)
            except:
                print(f'{FilterType} not present for {customerName}')


In [3]:
# # CODE BELOW NOT UPDATED - DEFERRED COMBINED DATASET WORK AS FIELDS ARE NOT CONSISTENT FOR EVERY CUSTOMER CAUSING MISSING DATA.

# #Combine and output as FS Audit Type main parquet
# for FilterType in FilterTypeList: 
#     dirPath = f'{config.data_location}//FS_audit_type_table_name//{FilterType}//'
#     fileName1 = f'{dirPath}FS_audit_type_table_name_{FilterType}.parquet'
#     if os.path.exists(fileName1):
#         os.remove(fileName1)
#     fileName2 = f'{dirPath}FS_audit_type_table_name_{FilterType}_count.parquet'
#     if os.path.exists(fileName2):
#         os.remove(fileName2)

#     files = glob.glob(f'{config.data_location}/FS_audit_type_table_name/{FilterType}/*.parquet') #{FilterType}
#     df = pd.concat([pd.read_parquet(fp) for fp in files])   
#     df.dropna(how='all', axis=1, inplace=True)
#     df['JSON_COUNT'] = df.filter(regex='_JSON').count(axis='columns') #Fragmented addition of JSON counts, replace with better method later.
#     df.to_parquet(path=fileName1)
    
#     df2 = df.groupby(['SOURCE_DB','SAMPLE_NUMBER','AUDIT_TIMESTAMP','TABLE_NAME','TABLE_KEY','T0083_SEVERITY']).count()
#     df2.to_parquet(path=fileName2)